<a href="https://colab.research.google.com/github/therishiraj/Agentic-AI-workshop/blob/main/day_6_fpd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q groq sentence-transformers numpy faiss-cpu \
               rank-bm25 streamlit

In [5]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ Ready" if GROQ_API_KEY else "❌ Add GROQ_API_KEY in Colab Secrets")

GROQ_MODEL = "llama-3.3-70b-versatile"   # our Groq model all day

✅ Ready


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
DIM = 384   # all-MiniLM-L6-v2 outputs 384 numbers per vector

# Making 50k REAL embeddings is slow, so we simulate a 50k corpus with random
# vectors of the SAME shape. FAISS treats them identically, so the SPEED
# comparison is honest. (Real text vectors would give even better accuracy.)
np.random.seed(0)
N = 50_000
corpus_vectors = np.random.random((N, DIM)).astype("float32")   # FAISS needs float32

# One real query embedding to search with
query = embedder.encode(["How much annual leave do employees get?"]).astype("float32")
print("Corpus:", corpus_vectors.shape, "| Query:", query.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Corpus: (50000, 384) | Query: (1, 384)


In [7]:
import faiss

# --- FLAT: exact brute-force ---
flat = faiss.IndexFlatL2(DIM)
flat.add(corpus_vectors)

# --- IVF: cluster-based (needs training to learn the clusters) ---
nlist = 200                                   # number of clusters (~sqrt(N))
quantizer = faiss.IndexFlatL2(DIM)
ivf = faiss.IndexIVFFlat(quantizer, DIM, nlist)
ivf.train(corpus_vectors)                     # 👈 IVF must be trained first
ivf.add(corpus_vectors)
ivf.nprobe = 10                               # how many clusters to search (speed vs accuracy)

# --- HNSW: graph-based ---
M = 32                                         # neighbours per node
hnsw = faiss.IndexHNSWFlat(DIM, M)
hnsw.hnsw.efConstruction = 40                  # build-time thoroughness
hnsw.hnsw.efSearch = 16                        # search-time thoroughness
hnsw.add(corpus_vectors)

print("All three indexes built. Vectors indexed:",
      flat.ntotal, ivf.ntotal, hnsw.ntotal)

All three indexes built. Vectors indexed: 50000 50000 50000


In [8]:
import time

def search(index, q, k=5):
    t0 = time.time()
    _, ids = index.search(q, k)
    ms = (time.time() - t0) * 1000
    return ids[0], ms

# Ground truth from the exact index
truth_ids, flat_ms = search(flat, query)
ivf_ids,  ivf_ms  = search(ivf, query)
hnsw_ids, hnsw_ms = search(hnsw, query)

def recall(approx, truth):
    return len(set(approx) & set(truth)) / len(truth)   # overlap with exact top-k

print(f"FLAT  | recall 1.00 (ground truth) | {flat_ms:6.2f} ms")
print(f"IVF   | recall {recall(ivf_ids, truth_ids):.2f}         | {ivf_ms:6.2f} ms")
print(f"HNSW  | recall {recall(hnsw_ids, truth_ids):.2f}         | {hnsw_ms:6.2f} ms")

FLAT  | recall 1.00 (ground truth) |  10.72 ms
IVF   | recall 0.20         |   0.76 ms
HNSW  | recall 0.00         |   0.54 ms


In [9]:
faiss.write_index(hnsw, "corpus.faiss")        # save to disk
reloaded = faiss.read_index("corpus.faiss")    # reload later without rebuilding
print("Reloaded vectors:", reloaded.ntotal)

Reloaded vectors: 50000


In [10]:
%%writefile app.py
import os
import streamlit as st
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

st.set_page_config(page_title="Ask Your Docs", page_icon="📖")

# --- cache heavy objects so they load once ---
@st.cache_resource
def get_embedder():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource
def get_groq():
    return Groq(api_key=os.environ["GROQ_API_KEY"])

embedder, groq_client = get_embedder(), get_groq()
MODEL = "llama-3.3-70b-versatile"

st.title("📖 Ask Your Documents")
st.caption("Upload a .txt file, then ask questions. Answers cite their source chunks.")

uploaded = st.file_uploader("Upload a text file", type=["txt"])

if uploaded:
    # 1) read + chunk (split on blank lines to keep it minimal)
    text = uploaded.read().decode("utf-8")
    chunks = [c.strip() for c in text.split("\n\n") if c.strip()]

    # 2) embed + index with FAISS (built once per upload)
    @st.cache_resource
    def build_index(chunks_tuple):
        vecs = embedder.encode(list(chunks_tuple)).astype("float32")
        idx = faiss.IndexFlatL2(vecs.shape[1])
        idx.add(vecs)
        return idx
    index = build_index(tuple(chunks))
    st.success(f"Indexed {len(chunks)} chunks. Ask away!")

    # 3) question box
    q = st.text_input("Your question")
    if q:
        qv = embedder.encode([q]).astype("float32")
        _, ids = index.search(qv, k=3)
        retrieved = [chunks[i] for i in ids[0]]
        context = "\n".join(retrieved)

        prompt = (f'Answer using ONLY the context. If absent, say "I don\'t know."\n'
                  f'Context:\n{context}\n\nQuestion: {q}')
        answer = groq_client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": prompt}],
            temperature=0).choices[0].message.content

        st.subheader("Answer")
        st.write(answer)

        with st.expander("📎 Source chunks used"):
            for i, chunk in enumerate(retrieved, 1):
                st.markdown(f"**Chunk {i}:** {chunk}")

Overwriting app.py


In [11]:
import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY   # already loaded from Colab Secrets earlier

In [12]:
# Download cloudflared (one-time)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start Streamlit in the background, then open a public tunnel to port 8501
!streamlit run app.py &>/content/logs.txt &
!sleep 5
!./cloudflared tunnel --url http://localhost:8501

2026-07-04T07:12:51Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-04T07:12:51Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-04T07:12:57Z INF +--------------------------------------------------------------------------------------------+
2026-07-04T07:12:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-04T07:12:57Z INF |  https://institutes-defining-houses-wyoming.trycloudfl

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
embedder = SentenceTransformer("all-MiniLM-L6-v2")

corpus = [
    "Acme Corp offers 24 days of paid annual leave to all full-time employees.",  # 0
    "Employees may work from home up to 3 days per week with manager approval.",   # 1
    "The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM.",          # 2
    "Acme reimburses home internet up to 1000 rupees per month for remote staff.", # 3
    "New joiners are on probation for the first 6 months of employment.",          # 4
]

# ground truth: question -> index of the chunk that answers it
gold = [
    ("How many leave days do I get?",        0),
    ("Can I work remotely?",                 1),
    ("When does the office open?",           2),
    ("What internet reimbursement is there?",3),
    ("How long is probation?",               4),
]

corpus_vecs = embedder.encode(corpus).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
def retrieve_ids(question, k):
    qv = embedder.encode([question]).astype("float32")[0]
    sims = corpus_vecs @ qv / (np.linalg.norm(corpus_vecs, axis=1) * np.linalg.norm(qv))
    return list(np.argsort(sims)[::-1][:k])

def recall_at_k(k):
    hits = sum(1 for q, correct in gold if correct in retrieve_ids(q, k))
    return hits / len(gold)

for k in (1, 2, 3):
    print(f"Recall@{k}: {recall_at_k(k):.2f}")

Recall@1: 1.00
Recall@2: 1.00
Recall@3: 1.00


In [15]:
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"

def generate(question, k=2):
    ids = retrieve_ids(question, k)
    context = "\n".join(corpus[i] for i in ids)
    prompt = f'Answer using ONLY this context.\nContext:\n{context}\n\nQ: {question}'
    ans = groq_client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0).choices[0].message.content
    return ans, context

def faithfulness(answer, context):
    judge = (f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
             "Is EVERY claim in the Answer supported by the Context? "
             "Reply with only YES or NO.")
    verdict = groq_client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": judge}],
        temperature=0).choices[0].message.content.strip().upper()
    return verdict.startswith("YES")

# Score the whole set
faithful = 0
for q, _ in gold:
    ans, ctx = generate(q)
    ok = faithfulness(ans, ctx)
    faithful += ok
    print(f"{'✅' if ok else '❌'}  {q}")
print(f"\nFaithfulness: {faithful}/{len(gold)} = {faithful/len(gold):.2f}")

✅  How many leave days do I get?
✅  Can I work remotely?
✅  When does the office open?
✅  What internet reimbursement is there?
❌  How long is probation?

Faithfulness: 4/5 = 0.80


In [16]:
syllabus = """Course: Introduction to Machine Learning (CS-201)

Grading: 40% assignments, 20% midterm, 40% final exam.

Attendance: A minimum of 75% attendance is required to sit the final exam.

Textbook: "Pattern Recognition and Machine Learning" by Bishop is the primary reference.

Office hours: Tuesdays and Thursdays, 2 to 4 PM, in Room 314.
"""
open("syllabus.txt", "w").write(syllabus)
print("Saved syllabus.txt — upload it to the app and ask 'What is the attendance requirement?'")

Saved syllabus.txt — upload it to the app and ask 'What is the attendance requirement?'


In [17]:
from rank_bm25 import BM25Okapi
import numpy as np

# --- BM25 (keyword) ---
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)

def bm25_scores(query):
    s = np.array(bm25.get_scores(query.lower().split()))
    return s / (s.max() + 1e-9)          # normalise to 0–1 for fair blending

# --- Dense (embeddings) ---
def dense_scores(query):
    qv = embedder.encode([query]).astype("float32")[0]
    s = corpus_vecs @ qv / (np.linalg.norm(corpus_vecs, axis=1) * np.linalg.norm(qv))
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

# --- Hybrid (weighted blend) ---
def hybrid_scores(query, alpha=0.5):
    return alpha * dense_scores(query) + (1 - alpha) * bm25_scores(query)

query = "MG Road office"      # a keyword-heavy query
for name, fn in [("BM25", bm25_scores), ("Dense", dense_scores), ("Hybrid", hybrid_scores)]:
    top = int(np.argmax(fn(query)))
    print(f"{name:6} → chunk {top}: {corpus[top]}")

BM25   → chunk 2: The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM.
Dense  → chunk 2: The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM.
Hybrid → chunk 2: The head office is at 42 MG Road, Bangalore, and opens at 9:00 AM.


In [18]:
out_of_corpus = [
    "What is Acme's annual revenue?",     # not in our chunks
    "Who is the CEO of Acme?",            # not in our chunks
    "What is the capital of France?",     # not even about Acme
]

for q in out_of_corpus:
    ans, ctx = generate(q)                # reuse Session 3's generate()
    refused = any(p in ans.lower() for p in ["don't know", "do not know",
                                             "not in the context", "no information"])
    print(f"{'✅ honest' if refused else '⚠️ hallucinated'}  |  Q: {q}\n   A: {ans}\n")

✅ honest  |  Q: What is Acme's annual revenue?
   A: There is no information provided about Acme's annual revenue.

✅ honest  |  Q: Who is the CEO of Acme?
   A: There is no information about the CEO of Acme in the provided context.

✅ honest  |  Q: What is the capital of France?
   A: There is no information about the capital of France in the given context.



In [19]:
def answer_with_gate(question, k=2, min_sim=0.25):
    qv = embedder.encode([question]).astype("float32")[0]
    sims = corpus_vecs @ qv / (np.linalg.norm(corpus_vecs, axis=1) * np.linalg.norm(qv))
    if sims.max() < min_sim:                       # nothing relevant enough
        return "I don't have information about that."
    return generate(question, k)[0]

print(answer_with_gate("What is the capital of France?"))  # → refuses, no LLM call needed
print(answer_with_gate("How many leave days do I get?"))   # → answers normally

I don't have information about that.
You get 24 days of paid annual leave.
